# CardioIA — MLP de ECG com base ampliada

Este notebook documenta e executa o experimento binário **normal × anormal** com as 491 imagens únicas auditadas. O código deixa visíveis a leitura, o pré-processamento, a divisão, a arquitetura, o treinamento e a avaliação. A Fase 1 permanece intacta e nenhuma saída possui validade clínica.

## 1. Protocolo e prevenção de vazamento

A divisão é estratificada pelas quatro classes originais. Primeiro reservamos 20% para teste final; depois separamos 20% do desenvolvimento para validação. O limiar é escolhido somente na validação. Hashes impedem que cópias exatas apareçam em mais de um conjunto, mas não substituem um identificador de paciente — indisponível na fonte.

In [1]:
from pathlib import Path
import os
import sys

os.environ.setdefault('TF_ENABLE_ONEDNN_OPTS', '0')

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from IPython.display import display
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, confusion_matrix,
    f1_score, precision_score, recall_score, roc_auc_score,
)
from sklearn.utils.class_weight import compute_class_weight

RAIZ = Path.cwd().resolve()
if RAIZ.name == 'notebooks':
    RAIZ = RAIZ.parents[1]
if not (RAIZ / 'fase2').exists():
    raise RuntimeError('Execute a partir da raiz do repositório ou de fase2/notebooks.')
sys.path.insert(0, str(RAIZ / 'fase2' / 'src'))

from treinar_mlp_ecg_ampliada import (
    dividir, inventario, modelo, pixels, sementes,
)
sementes()

## 2. Auditoria dos dados

O inventário exige exatamente 491 hashes originais distintos. Os derivados em tons de cinza tiveram cabeçalho administrativo, rodapé e metadados removidos antes do treinamento e da publicação dos exemplos.

In [2]:
dados = inventario()
auditoria = pd.DataFrame({
    'indicador': ['arquivos', 'hashes únicos', 'normais', 'anormais'],
    'valor': [
        len(dados), dados['hash_original'].nunique(),
        dados['alvo'].eq(0).sum(), dados['alvo'].eq(1).sum(),
    ],
})
display(auditoria)
display(dados.groupby(['classe_original', 'alvo']).size().rename('quantidade'))

,indicador,valor
0,arquivos,491
1,hashes únicos,491
2,normais,142
3,anormais,349


classe_original        alvo
abnormal_heartbeat     1       233
history_mi             1        86
myocardial_infarction  1        30
normal                 0       142
Name: quantidade, dtype: int64

## 3. Pré-processamento das imagens

Cada exame é convertido para escala de cinza, recebe autocontraste, preserva a proporção dentro de uma tela 96 × 56 e é normalizado para o intervalo [0, 1]. O fundo branco vira zero e o traçado escuro recebe valores maiores. Por fim, os pixels são achatados para entrada na MLP.

In [3]:
x = pixels(dados)
y = dados['alvo'].to_numpy(dtype=np.int32)
print(f'Matriz de entrada: {x.shape}; intervalo: {x.min():.3f} a {x.max():.3f}')

amostras = dados.groupby('classe_original', sort=True).head(1)
fig, eixos = plt.subplots(1, len(amostras), figsize=(14, 3))
for eixo, (indice, linha) in zip(eixos, amostras.iterrows()):
    eixo.imshow(x[indice].reshape(56, 96), cmap='gray_r', vmin=0, vmax=1)
    eixo.set_title(linha['classe_original'].replace('_', ' ').title())
    eixo.axis('off')
plt.tight_layout()

Matriz de entrada: (491, 5376); intervalo: 0.000 a 0.639


## 4. Treino, validação e teste

A função de divisão contém uma asserção de disjunção dos hashes. O teste permanece isolado até a avaliação final.

In [4]:
indices_treino, indices_validacao, indices_teste = dividir(dados, x)
conjuntos = {
    'treino': indices_treino, 'validação': indices_validacao, 'teste': indices_teste,
}
resumo_divisao = []
for nome, indices in conjuntos.items():
    parte = dados.iloc[indices]
    resumo_divisao.append({
        'conjunto': nome, 'n': len(indices),
        'normal': int(parte['alvo'].eq(0).sum()),
        'anormal': int(parte['alvo'].eq(1).sum()),
        'hashes_unicos': parte['hash_original'].nunique(),
    })
display(pd.DataFrame(resumo_divisao))

,conjunto,n,normal,anormal,hashes_unicos
0,treino,313,90,223,313
1,validação,79,23,56,79
2,teste,99,29,70,99


## 5. Arquitetura da MLP

A rede recebe 5.376 pixels, usa duas camadas densas e reduz sobreajuste com regularização L2, batch normalization e dropout. A saída sigmoide estima a probabilidade da classe anormal.

In [5]:
rede = modelo(x.shape[1])
rede.summary()

Model: "mlp_ecg_ampliada"
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │       344,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 16)             │         1,040 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 16)    

## 6. Treinamento

Os pesos são calculados apenas no treino para compensar o desbalanceamento observado. O `EarlyStopping` monitora a perda de validação e restaura os melhores pesos. Isso não transforma a amostra em estimativa de prevalência.

In [6]:
classes = np.unique(y[indices_treino])
pesos = compute_class_weight(
    class_weight='balanced', classes=classes, y=y[indices_treino]
)
pesos_classe = {int(classe): float(peso) for classe, peso in zip(classes, pesos)}
historico = rede.fit(
    x[indices_treino], y[indices_treino],
    validation_data=(x[indices_validacao], y[indices_validacao]),
    epochs=80, batch_size=16, class_weight=pesos_classe,
    callbacks=[tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=10, restore_best_weights=True
    )], verbose=0,
)
pd.DataFrame(historico.history)[['loss', 'val_loss']].plot(
    title='Perda no treino e na validação', figsize=(8, 4)
);
plt.xlabel('Época'); plt.ylabel('Binary cross-entropy'); plt.grid(alpha=.2)

## 7. Seleção do limiar sem consultar o teste

Em vez de assumir 0,5, testamos limiares de 0,20 a 0,80 somente na validação e escolhemos aquele com maior acurácia balanceada.

In [7]:
prob_validacao = rede.predict(x[indices_validacao], verbose=0).reshape(-1)
candidatos = np.linspace(.20, .80, 61)
scores = [
    balanced_accuracy_score(y[indices_validacao], prob_validacao >= limite)
    for limite in candidatos
]
limiar = float(candidatos[int(np.argmax(scores))])
print(f'Limiar selecionado na validação: {limiar:.2f}')

Limiar selecionado na validação: 0.77


## 8. Avaliação única no teste final

Relatamos acurácia, acurácia balanceada, precisão, recall, F1, ROC AUC e matriz de confusão. A classe positiva é `anormal`.

In [8]:
prob_teste = rede.predict(x[indices_teste], verbose=0).reshape(-1)
pred_teste = (prob_teste >= limiar).astype(int)
metricas = {
    'n_teste': len(indices_teste),
    'acuracia': accuracy_score(y[indices_teste], pred_teste),
    'acuracia_balanceada': balanced_accuracy_score(y[indices_teste], pred_teste),
    'precisao_anormal': precision_score(y[indices_teste], pred_teste, zero_division=0),
    'recall_anormal': recall_score(y[indices_teste], pred_teste, zero_division=0),
    'f1_anormal': f1_score(y[indices_teste], pred_teste, zero_division=0),
    'roc_auc': roc_auc_score(y[indices_teste], prob_teste),
}
display(pd.DataFrame([metricas]).round(4))
matriz = confusion_matrix(y[indices_teste], pred_teste, labels=[0, 1])
display(pd.DataFrame(
    matriz, index=['Real normal', 'Real anormal'],
    columns=['Predito normal', 'Predito anormal'],
))

,n_teste,acuracia,acuracia_balanceada,precisao_anormal,recall_anormal,f1_anormal,roc_auc
0,99,0.697,0.7453,0.9167,0.6286,0.7458,0.8241


,Predito normal,Predito anormal
Real normal,25,4
Real anormal,26,44


## 9. Resultado e interpretação responsável

Com semente 42, TensorFlow CPU 2.21.0, Keras 3.12.0, operações determinísticas e oneDNN desativado, o teste de 99 imagens obteve aproximadamente 69,7% de acurácia, 74,5% de acurácia balanceada, 91,7% de precisão anormal, 62,9% de recall anormal, 74,6% de F1 anormal e ROC AUC 0,824. A matriz foi `[[25, 4], [26, 44]]`. A melhora frente ao baseline de 60 imagens reforça a importância do tamanho amostral, mas ainda há 26 falsos negativos.

Limitações: a fonte não fornece chave confiável de paciente; hashes detectam apenas duplicatas exatas; as classes não representam prevalência clínica; não houve validação externa ou prospectiva. Portanto, o experimento é exclusivamente acadêmico e não pode apoiar diagnóstico ou conduta.